In [ ]:
import sys;sys.path.append('../src/')
import os

from models.unet import UNet
from models.ddpm import DDPM
from data import DogSet
from torch.utils.data import DataLoader
import torch
from torchvision.utils import make_grid
from data import denormalize
from torchvision.transforms.functional import to_pil_image

os.makedirs('../samples/', exist_ok=True)

ds = DogSet(data_dir='../data/afhq',image_size=64, hflip=False)
loader = DataLoader(dataset=ds, batch_size=4,shuffle=False)

batch = next(iter(loader))
assert batch.shape == (4,3,64,64)
torch.manual_seed(4)

unet = UNet(in_channels =3,
                base_channel = 64,
                time_emb_dim = 64,
                channels_mults = (1,2,2,4),
                num_res_blocks = 2,
                num_groups =32)

ddpm = DDPM(unet= unet, T=1000)
loss = ddpm(batch)

print("loss:", loss.item(), "| scalar:", loss.dim()==0)
loss.backward()

got = [n for n,p in ddpm.named_parameters() if p.grad is not None and p.grad.abs().sum()>0]
print("params with nonzero grad:", len(got), "/", sum(1 for _ in ddpm.named_parameters()))

print("buffers:", [n for n,_ in ddpm.named_buffers()])
print("total params:", sum(p.numel() for p in ddpm.parameters()))




loss: 1.0784989595413208 | scalar: True
params with nonzero grad: 380 / 380
buffers: ['betas', 'alphas', 'alphas_hat', 'sqrt_alphas_hat', 'sqrt_one_minus_alphas_hat']
total params: 18536323


In [8]:
from torchvision.utils import save_image
from models.ddpm import load_ddpm_model

model = load_ddpm_model(path='../checkpoints/ddpm_best.pt',device='mps')
model.eval()
samples = model.sample(batch_size=16, image_size= (64,64))
save_image(denormalize(samples.cpu()), '../samples/ddpm_best.png',nrow = 4)


In [ ]:

model =load_ddpm_model(path= '../checkpoints/ddpm_last.pt', device='mps')
model.eval()
samples = model.sample(batch_size=16, image_size=(64,64))
save_image(denormalize(samples.cpu()), '../samples/ddpm_last.png',nrow = 4)